In [ ]:
!pip install -q ultralytics

!ls /kaggle/input/datasets/limmaximus/yolo-club

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

# Install available mediapipe version
!pip install mediapipe==0.10.14 -q

In [ ]:
# =============================================================================
# SUPPRESS WARNINGS & INSTALL DEPENDENCIES
# =============================================================================
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_MIN_LOG_LEVEL'] = '3'



# =============================================================================
# COMPLETE GOLF SWING ANALYSIS WITH YOLO CLUB DETECTION
# =============================================================================

import cv2
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, savgol_filter
import matplotlib.pyplot as plt

# Import mediapipe after fresh install
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# =============================================================================
# SETTINGS - UPDATE THESE PATHS
# =============================================================================
VIDEO_PATH = "/kaggle/input/datasets/limmaximus/sampleswing3/sample_swing3.mp4"
CLUB_MODEL_PATH = "/kaggle/input/datasets/limmaximus/trained-model/best.pt"

CONF = 0.25
CLUB_KEYPOINT_IDX = 0

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def safe_savgol(series, default_window=7, poly=2):
    """Safely apply Savitzky-Golay smoothing."""
    s = pd.Series(series).astype(float).ffill().bfill()
    n = len(s)
    if n < 5:
        return s.values
    w = min(default_window, n)
    if w % 2 == 0:
        w -= 1
    if w < 5:
        return s.values
    poly = min(poly, w - 1)
    try:
        return savgol_filter(s.values, window_length=w, polyorder=poly, mode="nearest")
    except Exception:
        return s.values


def rolling_std(x, win=20):
    return pd.Series(x).rolling(win, center=True).std().bfill().ffill().values

def interp_nan_limited(arr, limit=5):
    """
    Interpolate only short NaN gaps (<= limit). Do NOT ffill/bfill entire clip.
    This prevents inventing a smooth curve where detections are missing.
    """
    s = pd.Series(arr).astype(float)
    s = s.interpolate(limit=limit, limit_direction="both")
    return s.values
    
def extract_frame(video_path, frame_idx):
    """Extract a single frame from video."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if ret:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return None


def angle_3pt(p1, p2, p3):
    """Angle at p2 formed by p1-p2-p3 (in degrees)."""
    v1 = np.array(p1) - np.array(p2)
    v2 = np.array(p3) - np.array(p2)
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 < 1e-8 or n2 < 1e-8:
        return 180.0
    cosang = np.dot(v1, v2) / (n1 * n2)
    cosang = np.clip(cosang, -1, 1)
    return float(np.degrees(np.arccos(cosang)))


# =============================================================================
# STEP 1: MEDIAPIPE POSE EXTRACTION (Using legacy API)
# =============================================================================
def extract_pose_mediapipe(video_path):
    """Extract pose landmarks using MediaPipe."""
    
    print("=" * 60)
    print("STEP 1: MEDIAPIPE POSE EXTRACTION")
    print("=" * 60)
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Cannot open video: {video_path}")
        return None, None
    
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Video: {width}x{height} @ {fps:.1f} FPS, {total_frames} frames")
    
    # Try legacy API first
    try:
        mp_pose = mp.solutions.pose
        pose = mp_pose.Pose(
            static_image_mode=False,
            model_complexity=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        use_legacy = True
        print("Using MediaPipe legacy API")
    except AttributeError:
        print("Legacy API not available, using tasks API")
        use_legacy = False
        
        # Download pose landmarker model
        !wget -q -O /tmp/pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task
        
        base_options = python.BaseOptions(model_asset_path='/tmp/pose_landmarker.task')
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            output_segmentation_masks=False
        )
        pose = vision.PoseLandmarker.create_from_options(options)
    
    pose_data = []
    frame_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        row = {"frame": frame_idx}
        
        if use_legacy:
            results = pose.process(frame_rgb)
            
            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                
                # Left wrist (index 15)
                row["wrist_x"] = landmarks[15].x
                row["wrist_y"] = landmarks[15].y
                row["wrist_visibility"] = landmarks[15].visibility
                
                # Left elbow (index 13)
                row["elbow_x"] = landmarks[13].x
                row["elbow_y"] = landmarks[13].y
                row["elbow_visibility"] = landmarks[13].visibility
                
                # Left shoulder (index 11)
                row["shoulder_x"] = landmarks[11].x
                row["shoulder_y"] = landmarks[11].y
                row["shoulder_visibility"] = landmarks[11].visibility
                
                # Left hip (index 23)
                row["hip_x"] = landmarks[23].x
                row["hip_y"] = landmarks[23].y
                row["hip_visibility"] = landmarks[23].visibility
        else:
            # Tasks API
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            results = pose.detect(mp_image)
            
            if results.pose_landmarks and len(results.pose_landmarks) > 0:
                landmarks = results.pose_landmarks[0]
                
                # Left wrist (index 15)
                row["wrist_x"] = landmarks[15].x
                row["wrist_y"] = landmarks[15].y
                row["wrist_visibility"] = landmarks[15].visibility
                
                # Left elbow (index 13)
                row["elbow_x"] = landmarks[13].x
                row["elbow_y"] = landmarks[13].y
                row["elbow_visibility"] = landmarks[13].visibility
                
                # Left shoulder (index 11)
                row["shoulder_x"] = landmarks[11].x
                row["shoulder_y"] = landmarks[11].y
                row["shoulder_visibility"] = landmarks[11].visibility
                
                # Left hip (index 23)
                row["hip_x"] = landmarks[23].x
                row["hip_y"] = landmarks[23].y
                row["hip_visibility"] = landmarks[23].visibility
        
        pose_data.append(row)
        frame_idx += 1
        
        if frame_idx % 50 == 0:
            print(f"  Processed {frame_idx}/{total_frames} frames...")
    
    cap.release()
    
    if use_legacy:
        pose.close()
    
    df = pd.DataFrame(pose_data)
    
    # Detection rate
    if "wrist_visibility" in df.columns:
        detection_rate = df["wrist_visibility"].notna().sum() / len(df) * 100
    else:
        detection_rate = 0
    
    print(f"MediaPipe detection rate: {detection_rate:.1f}%")
    print(f"Extracted {len(df)} frames")
    
    return df, fps


# =============================================================================
# STEP 2: YOLO CLUB DETECTION
# =============================================================================
def extract_club_trajectory(video_path, model_path, conf=0.25, keypoint_idx=0):
    """Extract club head trajectory using YOLO. Supports pose keypoints or bbox fallback."""

    print("\n" + "=" * 60)
    print("STEP 2: YOLO CLUB DETECTION")
    print("=" * 60)

    from ultralytics import YOLO

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Cannot open video: {video_path}")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print(f"Loading YOLO model: {model_path}")
    model = YOLO(model_path)
    print(f"Model classes: {model.names}")

    cap = cv2.VideoCapture(video_path)

    y_values = []
    conf_values = []
    valid_values = []

    frame_idx = 0
    print("Processing video with YOLO...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        res = model.predict(frame, conf=conf, verbose=False)[0]

        y_val = np.nan
        c_val = 0.0
        valid = 0

        # --- Pose keypoints path ---
        if res.keypoints is not None and len(res.keypoints) > 0 and res.boxes is not None and len(res.boxes) > 0:
            best_idx = int(np.argmax(res.boxes.conf.cpu().numpy()))
            c_box = float(res.boxes.conf[best_idx].cpu().numpy())
            kpts = res.keypoints.xy[best_idx].cpu().numpy()

            if keypoint_idx < len(kpts):
                y_val = float(kpts[keypoint_idx, 1])
                c_val = c_box
                valid = 1

        # --- Detection bbox fallback ---
        elif res.boxes is not None and len(res.boxes) > 0:
            best_idx = int(np.argmax(res.boxes.conf.cpu().numpy()))
            c_box = float(res.boxes.conf[best_idx].cpu().numpy())
            box = res.boxes.xyxy[best_idx].cpu().numpy()
            y_val = float(box[3])  # y2 (bottom of bbox) ~ lowest point
            c_val = c_box
            valid = 1

        y_values.append(y_val)
        conf_values.append(c_val)
        valid_values.append(valid)

        frame_idx += 1
        if frame_idx % 50 == 0:
            print(f"  Processed {frame_idx}/{total_frames} frames...")

    cap.release()

    y_raw = np.array(y_values, dtype=float)
    c_raw = np.array(conf_values, dtype=float)
    valid = np.array(valid_values, dtype=int)

    detection_rate = np.sum(valid == 1) / len(valid) * 100
    print(f"YOLO detection rate: {detection_rate:.1f}%")

    # Interpolate only short gaps, then smooth
    y_interp = interp_nan_limited(y_raw, limit=5)
    y_smooth = safe_savgol(y_interp, default_window=9, poly=2)

    return {
        "y_raw": y_raw,
        "y_smooth": y_smooth,
        "conf_raw": c_raw,
        "valid": valid,
        "detection_rate": detection_rate,
        "fps": fps,
        "total_frames": len(y_raw)
    }
# =============================================================================
# STEP 3: PHASE DETECTION
# =============================================================================
def detect_phases(pose_df, club_data, fps, debug=True):
    """
    Detect swing phases:
    - Address, Backswing Start, Backswing Top: MediaPipe wrist_y
    - Impact: YOLO club lowest point AFTER top (with valid gating), else wrist fallback
    - Finish: wrist speed stabilisation (same as your logic)
    """

    print("\n" + "=" * 60)
    print("STEP 3: PHASE DETECTION")
    print("=" * 60)

    # -----------------------------
    # Smooth wrist data
    # -----------------------------
    for col in ["wrist_x", "wrist_y", "elbow_x", "elbow_y", "shoulder_x", "shoulder_y", "hip_x", "hip_y"]:
        if col in pose_df.columns:
            pose_df[f"{col}_smooth"] = safe_savgol(pose_df[col].values, default_window=7, poly=2)

    if "wrist_y_smooth" not in pose_df.columns:
        print("ERROR: No wrist data")
        return None

    y = pose_df["wrist_y_smooth"].values.astype(float)
    n = len(y)

    # =========================================
    # 2) ADDRESS + BACKSWING START (your same logic)
    # =========================================
    MIN_TOTAL_DROP = 0.2

    total_range = np.nanpercentile(y, 95) - np.nanpercentile(y, 5)
    total_range = max(float(total_range), 1e-6)
    significant_drop = max(0.30 * total_range, MIN_TOTAL_DROP)

    win = 20
    rs = rolling_std(y, win=win)

    max_search = min(n // 2, 250)
    stability_threshold = 0.06 * total_range
    min_len = 12

    stable_segments = []
    start = None
    length = 0

    for i in range(win, max_search):
        if rs[i] < stability_threshold:
            if start is None:
                start = i
            length += 1
        else:
            if start is not None and length >= min_len:
                stable_segments.append((start, i - 1, length))
            start = None
            length = 0

    if start is not None and length >= min_len:
        stable_segments.append((start, max_search - 1, length))

    if stable_segments:
        stable_segments.sort(key=lambda t: t[2], reverse=True)
        s0, s1, _ = stable_segments[0]
    else:
        s0, s1 = 0, min(20, max_search - 1)

    address_idx = s0
    address_y = float(np.nanmean(y[s0:s1 + 1]))

    local_drop = 0.02 * total_range
    confirm_window = 40
    backswing_start = s1

    for i in range(s1, min(s1 + 80, n - confirm_window - 1)):
        if y[i] < address_y - local_drop:
            future_min = float(np.nanmin(y[i:i + confirm_window]))
            if (address_y - future_min) >= significant_drop:
                backswing_start = max(i - 5, s1)
                break

    if backswing_start == s1:
        dy = np.diff(y)
        for i in range(20, min(250, len(dy) - 6)):
            if np.all(dy[i:i + 6] < -0.01 * total_range):
                backswing_start = i
                break

    if debug:
        print(f"Address: frame {address_idx}, address_y: {address_y:.4f}")
        print(f"Backswing start: frame {backswing_start}")
    # =========================================
    # 3) BACKSWING TOP (FIXED: velocity sign change)
    # =========================================
    if backswing_start >= n - 10:
        print("ERROR: Backswing start too late")
        return None

    # Work on a window after backswing start
    # y decreases when wrist goes UP (because image coords y down)
    y_seg = y[backswing_start: min(backswing_start + 220, n)].copy()

    # Derivative (velocity proxy)
    dy = np.diff(y_seg)

    # We want: going UP => dy < 0, then start going DOWN => dy > 0
    # Use "sustained" sign change to avoid noise
    sustain = 6  # number of consecutive frames
    top_local = None

    for i in range(10, len(dy) - sustain):
        was_up = np.all(dy[i - sustain:i] < 0)
        now_down = np.all(dy[i:i + sustain] > 0)
        if was_up and now_down:
            top_local = i
            break

    if top_local is None:
        # Fallback: pick global min y in this window (highest wrist)
        top_local = int(np.nanargmin(y_seg))

    backswing_top = int(backswing_start + top_local)

    if debug:
        print(f"Backswing top (velocity): frame {backswing_top}, y: {y[backswing_top]:.4f}")

def detect_impact_from_yolo_club(club_data, backswing_top, fps, debug=False):
    """
    Impact from YOLO clubhead Y:
    - After top, find first sustained "drop" (club y increasing fast)
    - Then take the first local bottom (local max in club_y)
    Uses club_data["y_smooth"] and club_data["valid"].
    """

    if club_data is None or "y_smooth" not in club_data:
        return None, "no_club_series"

    club_y = np.asarray(club_data["y_smooth"], dtype=float)
    valid = np.asarray(club_data.get("valid", np.ones_like(club_y, dtype=int)), dtype=int)
    n = len(club_y)

    top = int(np.clip(backswing_top, 0, n - 1))

    if valid[top:].sum() < 8:
        return None, "club_too_sparse"

    # window after top (time-based)
    start = top + int(0.05 * fps)
    end   = min(n, top + int(1.2 * fps))

    if end <= start + 10:
        return None, "window_too_small"

    yseg = club_y[start:end].copy()
    vseg = valid[start:end].copy().astype(bool)

    # interpolate only short gaps inside this window
    yseg2 = pd.Series(yseg).where(vseg).interpolate(limit=5).values
    dy = np.diff(yseg2)

    # robust threshold for a "real drop"
    dy_med = float(np.nanmedian(dy))
    dy_mad = float(np.nanmedian(np.abs(dy - dy_med))) + 1e-6
    drop_thr = dy_med + 3.0 * dy_mad

    sustain = 4
    drop_start_off = None
    for i in range(5, len(dy) - sustain):
        if np.all(dy[i:i+sustain] > drop_thr):
            drop_start_off = i
            break

    if drop_start_off is None:
        # fallback: lowest club in window (max y) among valid
        idxs = np.where(vseg)[0]
        if len(idxs) == 0:
            return None, "no_valid_in_window"
        best = int(idxs[np.argmax(yseg[idxs])])
        return int(start + best), "yolo_lowest_fallback"

    drop_start = start + drop_start_off

    # now pick the FIRST local bottom after drop_start (local max in y)
    bottom_win = int(0.45 * fps)
    lo = drop_start
    hi = min(n, drop_start + bottom_win)

    seg = club_y[lo:hi].copy()
    segv = valid[lo:hi].astype(bool)

    idxs = np.where(segv)[0]
    if len(idxs) < 5:
        return int(drop_start), "drop_start_only"

    seg2 = pd.Series(seg).where(segv).interpolate(limit=5).values
    peaks, _ = find_peaks(seg2, distance=max(4, int(0.08 * fps)))

    if len(peaks) > 0:
        impact = int(lo + peaks[0])
        if debug:
            print("YOLO impact:", impact, "(drop_start:", drop_start, ")")
        return impact, "yolo_drop_then_bottom"

    best = int(idxs[np.argmax(seg[idxs])])
    # =========================================
    # 5) FINISH DETECTION (your same logic)
    # =========================================
    finish = n - 1

    if "wrist_x_smooth" in pose_df.columns:
        wrist_x = pose_df["wrist_x_smooth"].values.astype(float)
        wrist_y = pose_df["wrist_y_smooth"].values.astype(float)
        dt = 1.0 / fps

        vx = np.diff(wrist_x) / dt
        vy = np.diff(wrist_y) / dt
        speed = np.sqrt(vx * vx + vy * vy)

        post_impact_speed = speed[min(impact, len(speed) - 1):]

        if len(post_impact_speed) >= 20:
            peak_speed = float(np.nanmax(speed))
            thresh = 0.12 * max(peak_speed, 1e-6)
            stable_len = 12
            count = 0

            for i in range(len(post_impact_speed)):
                if post_impact_speed[i] < thresh:
                    count += 1
                    if count >= stable_len:
                        finish = min(impact + i, n - 1)
                        break
                else:
                    count = 0

    # =========================================
    # PRINT RESULTS
    # =========================================
    print(f"\n{'='*50}")
    print("PHASE DETECTION RESULTS:")
    print(f"{'='*50}")
    print(f"  Address:         Frame {address_idx} ({address_idx/fps:.2f}s)")
    print(f"  Backswing Start: Frame {backswing_start} ({backswing_start/fps:.2f}s)")
    print(f"  Backswing Top:   Frame {backswing_top} ({backswing_top/fps:.2f}s)")
    print(f"  Impact:          Frame {impact} ({impact/fps:.2f}s) [{impact_method}]")
    print(f"  Finish:          Frame {finish} ({finish/fps:.2f}s)")
    print(f"{'='*50}")

    return {
        "address_idx": address_idx,
        "backswing_start_idx": backswing_start,
        "backswing_top_idx": backswing_top,
        "impact_idx": impact,
        "impact_method": impact_method,
        "finish_idx": finish,
        "fps": fps
    }

# =============================================================================
# STEP 5: VISUALIZATION
# =============================================================================
def visualize_results(video_path, pose_df, club_data, phases, metrics, fps):
    """Visualize results."""
    
    print("\n" + "=" * 60)
    print("STEP 5: VISUALIZATION")
    print("=" * 60)
    
    phase_frames = {
        "Address": phases["address_idx"],
        "Backswing Top": phases["backswing_top_idx"],
        "Impact": phases["impact_idx"],
        "Finish": phases["finish_idx"]
    }
    
    # Phase images
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    for i, (name, frame_idx) in enumerate(phase_frames.items()):
        img = extract_frame(video_path, frame_idx)
        if img is not None:
            axes[i].imshow(img)
            axes[i].set_title(f"{name}\nFrame {frame_idx} ({frame_idx/fps:.2f}s)", fontsize=12)
        else:
            axes[i].set_title(f"{name}\nFrame {frame_idx} (failed)", fontsize=12)
        axes[i].axis("off")
    
    plt.suptitle(f"Swing Phases (Impact method: {phases['impact_method']})", fontsize=14)
    plt.tight_layout()
    plt.savefig("/kaggle/working/phase_frames.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    # Trajectories
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    wrist_y = pose_df["wrist_y_smooth"].values
    frames = np.arange(len(wrist_y))
    
    axes[0].plot(frames, wrist_y, label="Wrist Y (MediaPipe)", linewidth=2, color="#3498db")
    axes[0].axvline(phases["address_idx"], color="#27ae60", linestyle="--", label=f"Address")
    axes[0].axvline(phases["backswing_top_idx"], color="#f39c12", linestyle="--", label=f"Backswing Top")
    axes[0].axvline(phases["impact_idx"], color="#e74c3c", linestyle="--", label=f"Impact")
    axes[0].axvline(phases["finish_idx"], color="#9b59b6", linestyle="--", label=f"Finish")
    axes[0].set_xlabel("Frame")
    axes[0].set_ylabel("Wrist Y")
    axes[0].set_title("Wrist Trajectory (MediaPipe)")
    axes[0].legend(loc="upper right")
    axes[0].grid(True, alpha=0.3)
    axes[0].invert_yaxis()
    
    if club_data is not None:
        club_y = club_data["y_smooth"]
        axes[1].plot(np.arange(len(club_y)), club_y, label="Club Y (YOLO)", linewidth=2, color="#e74c3c")
        axes[1].axvline(phases["backswing_top_idx"], color="#f39c12", linestyle="--", label=f"Backswing Top")
        axes[1].axvline(phases["impact_idx"], color="#e74c3c", linestyle="--", label=f"Impact")
        axes[1].set_xlabel("Frame")
        axes[1].set_ylabel("Club Y (pixels)")
        axes[1].set_title(f"Club Trajectory (YOLO) - Detection rate: {club_data['detection_rate']:.1f}%")
        axes[1].legend(loc="upper right")
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, "Club detection not available", ha="center", va="center", fontsize=14)
    
    plt.tight_layout()
    plt.savefig("/kaggle/working/trajectories.png", dpi=150)
    plt.show()
    
    print("\nSaved: /kaggle/working/phase_frames.png")
    print("Saved: /kaggle/working/trajectories.png")


# =============================================================================
# MAIN EXECUTION
# =============================================================================
print("=" * 60)
print("GOLF SWING ANALYSIS")
print("=" * 60)
print(f"Video: {VIDEO_PATH}")
print(f"Club Model: {CLUB_MODEL_PATH}")
print("=" * 60)

# Step 1: MediaPipe
pose_df, fps = extract_pose_mediapipe(VIDEO_PATH)

if pose_df is not None and len(pose_df) > 0:
    
    # Step 2: YOLO
    club_data = extract_club_trajectory(VIDEO_PATH, CLUB_MODEL_PATH, conf=CONF, keypoint_idx=CLUB_KEYPOINT_IDX)
    
    # Step 3: Phases
    phases = detect_phases(pose_df, club_data, fps, debug=True)
    
    if phases is not None:
        # Step 4: Metrics
        metrics = calculate_metrics(pose_df, phases, fps)
        
        # Step 5: Visualize
        visualize_results(VIDEO_PATH, pose_df, club_data, phases, metrics, fps)
        
        # Summary
        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(f"Impact Method: {phases['impact_method']}")
        print(f"Tempo: {metrics['tempo_ratio']}:1 ({metrics['tempo_label']})")
        if metrics['elbow_angle']:
            print(f"Elbow Angle: {metrics['elbow_angle']:.1f}° ({metrics['arm_label']})")
        print(f"Speed Timing: {metrics['speed_label']}")
        print(f"Overall: {metrics['overall_score']}/100 ({metrics['overall_rating']})")
        print("=" * 60)
else:
    print("ERROR: Could not extract pose data")

In [ ]:
!yolo task=pose mode=train \
  model=yolov8n-pose.pt \
  data=/kaggle/input/datasets/limmaximus/yolo-club/data.yaml \
  imgsz=640 \
  epochs=80 \
  batch=16

In [ ]:
!yolo task=pose mode=predict \
  model=/kaggle/working/runs/pose/train2/weights/best.pt \
  source=/kaggle/input/datasets/limmaximus/testing/pro_swing.mp4 \
  conf=0.30

In [ ]:
# =============================================================================
# KAGGLE FULL NOTEBOOK: MediaPipe phases + YOLO impact refinement (±5 frames)
# NO ROBOFLOW API. Uses your MediaPipe logic for address/start/top/impact-wrist,
# then refines impact with YOLO by searching the lowest club point (max y) within
# ±5 frames of the wrist-impact.
# =============================================================================

# -----------------------------
# Install deps (run once)
# -----------------------------
!pip -q install ultralytics mediapipe==0.10.9 opencv-python-headless numpy pandas scipy matplotlib

# -----------------------------
# Imports + suppress noise
# -----------------------------
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter

import mediapipe as mp
from ultralytics import YOLO

# =============================================================================
# SETTINGS - UPDATE THESE PATHS
# =============================================================================
VIDEO_PATH = "/kaggle/input/datasets/limmaximus/testing/pro_swing.mp4"
CLUB_MODEL_PATH = "/kaggle/input/datasets/limmaximus/trained-model/best.pt"

# YOLO settings
CONF = 0.25
CLUB_KEYPOINT_IDX = 0      # if your model is pose w/ keypoints; 0 = clubhead
IMPACT_WINDOW_FRAMES = 5   # +/- window around wrist impact

# =============================================================================
# Helpers
# =============================================================================
def safe_savgol(series, default_window=7, poly=2):
    """Safely apply Savitzky-Golay smoothing (handles NaNs)."""
    s = pd.Series(series).astype(float).ffill().bfill()
    n = len(s)
    if n < 5:
        return s.values
    w = min(default_window, n)
    if w % 2 == 0:
        w -= 1
    if w < 5:
        return s.values
    poly = min(poly, w - 1)
    try:
        return savgol_filter(s.values, window_length=w, polyorder=poly, mode="nearest")
    except Exception:
        return s.values

def rolling_std(x, win=20):
    return pd.Series(x).rolling(win, center=True).std().bfill().ffill().values

def interp_nan_limited(arr, limit=5):
    """Interpolate only short NaN gaps."""
    s = pd.Series(arr).astype(float)
    s = s.interpolate(limit=limit, limit_direction="both")
    return s.values

def extract_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

def angle_3pt(p1, p2, p3):
    """Angle at p2 formed by p1-p2-p3 (in degrees)."""
    v1 = np.array(p1) - np.array(p2)
    v2 = np.array(p3) - np.array(p2)
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 < 1e-8 or n2 < 1e-8:
        return 180.0
    cosang = np.dot(v1, v2) / (n1 * n2)
    cosang = np.clip(cosang, -1, 1)
    return float(np.degrees(np.arccos(cosang)))

# =============================================================================
# STEP 1: MediaPipe pose extraction (legacy Pose)
# =============================================================================
def extract_pose_mediapipe(video_path):
    print("=" * 60)
    print("STEP 1: MEDIAPIPE POSE EXTRACTION")
    print("=" * 60)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Cannot open video: {video_path}")
        return None, None

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video: {w}x{h} @ {fps:.2f} FPS, {total_frames} frames")

    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    rows = []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(frame_rgb)

        row = {"frame": frame_idx}
        if res.pose_landmarks:
            lm = res.pose_landmarks.landmark
            # Left side: 11 shoulder, 13 elbow, 15 wrist, 23 hip
            row["shoulder_x"], row["shoulder_y"], row["shoulder_vis"] = lm[11].x, lm[11].y, lm[11].visibility
            row["elbow_x"],    row["elbow_y"],    row["elbow_vis"]    = lm[13].x, lm[13].y, lm[13].visibility
            row["wrist_x"],    row["wrist_y"],    row["wrist_vis"]    = lm[15].x, lm[15].y, lm[15].visibility
            row["hip_x"],      row["hip_y"],      row["hip_vis"]      = lm[23].x, lm[23].y, lm[23].visibility

        rows.append(row)

        frame_idx += 1
        if frame_idx % 50 == 0:
            print(f"  Processed {frame_idx}/{total_frames} frames...")

    cap.release()
    pose.close()

    df = pd.DataFrame(rows)
    det_rate = df["wrist_vis"].notna().mean() * 100 if "wrist_vis" in df.columns else 0.0
    print(f"MediaPipe detection rate: {det_rate:.1f}%")
    print(f"Extracted {len(df)} frames\n")
    return df, float(fps)

# =============================================================================
# STEP 2: YOLO club trajectory (pose keypoints OR bbox fallback)
# =============================================================================
def extract_club_trajectory(video_path, model_path, conf=0.25, keypoint_idx=0):
    print("\n" + "=" * 60)
    print("STEP 2: YOLO CLUB DETECTION")
    print("=" * 60)

    model = YOLO(model_path)
    print(f"Loading YOLO model: {model_path}")
    print(f"Model classes: {model.names}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Cannot open video: {video_path}")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    y_values = []
    conf_values = []
    valid_values = []

    frame_idx = 0
    print("Processing video with YOLO...")
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        res = model.predict(frame, conf=conf, verbose=False)[0]

        y_val = np.nan
        c_val = 0.0
        valid = 0

        # Pose model path
        if res.keypoints is not None and len(res.keypoints) > 0 and res.boxes is not None and len(res.boxes) > 0:
            best_idx = int(np.argmax(res.boxes.conf.cpu().numpy()))
            c_box = float(res.boxes.conf[best_idx].cpu().numpy())
            kpts = res.keypoints.xy[best_idx].cpu().numpy()
            if keypoint_idx < len(kpts):
                y_val = float(kpts[keypoint_idx, 1])
                c_val = c_box
                valid = 1

        # Detection fallback path (use bottom of bbox)
        elif res.boxes is not None and len(res.boxes) > 0:
            best_idx = int(np.argmax(res.boxes.conf.cpu().numpy()))
            c_box = float(res.boxes.conf[best_idx].cpu().numpy())
            box = res.boxes.xyxy[best_idx].cpu().numpy()
            y_val = float(box[3])  # y2 bottom
            c_val = c_box
            valid = 1

        y_values.append(y_val)
        conf_values.append(c_val)
        valid_values.append(valid)

        frame_idx += 1
        if frame_idx % 50 == 0:
            print(f"  Processed {frame_idx}/{total_frames} frames...")

    cap.release()

    y_raw = np.array(y_values, dtype=float)
    c_raw = np.array(conf_values, dtype=float)
    valid = np.array(valid_values, dtype=int)

    det_rate = valid.mean() * 100
    print(f"YOLO detection rate: {det_rate:.1f}%")

    # interpolate short gaps then smooth
    y_interp = interp_nan_limited(y_raw, limit=5)
    y_smooth = safe_savgol(y_interp, default_window=9, poly=2)

    return {
        "y_raw": y_raw,
        "y_smooth": y_smooth,
        "conf_raw": c_raw,
        "valid": valid,
        "detection_rate": float(det_rate),
        "fps": float(fps),
        "total_frames": len(y_raw),
    }

# =============================================================================
# Impact refinement: YOLO search within +/- window frames around wrist impact
# =============================================================================
def refine_impact_with_yolo_window(club_data, impact_wrist, window=5, debug=False):
    """
    Refine impact using YOLO club Y around a wrist-estimated impact.
    Picks the lowest club point (max y) within +/- window frames.
    """
    if club_data is None or "y_smooth" not in club_data:
        return None, "no_club"

    club_y = np.asarray(club_data["y_smooth"], dtype=float)
    valid  = np.asarray(club_data.get("valid", np.ones_like(club_y, dtype=int)), dtype=int)
    n = len(club_y)

    c = int(np.clip(impact_wrist, 0, n - 1))
    lo = max(0, c - window)
    hi = min(n, c + window + 1)

    seg = club_y[lo:hi]
    segv = valid[lo:hi]

    idxs = np.where(segv == 1)[0]
    if len(idxs) == 0:
        return None, "no_valid_in_window"

    best_off = int(idxs[np.argmax(seg[idxs])])  # max y = lowest club
    impact_yolo = int(lo + best_off)

    if debug:
        print(f"YOLO refine window [{lo},{hi}) picked {impact_yolo} (wrist={impact_wrist})")

    return impact_yolo, "yolo_refine_pm_window"

# =============================================================================
# Phase detection (keeps MediaPipe logic; YOLO only refines impact)
# =============================================================================
def detect_phases(pose_df, club_data, fps, debug=True):
    print("\n" + "=" * 60)
    print("STEP 3: PHASE DETECTION")
    print("=" * 60)

    # Smooth MediaPipe coords
    for col in ["wrist_x","wrist_y","elbow_x","elbow_y","shoulder_x","shoulder_y","hip_x","hip_y"]:
        if col in pose_df.columns:
            pose_df[f"{col}_smooth"] = safe_savgol(pose_df[col].values, default_window=7, poly=2)

    if "wrist_y_smooth" not in pose_df.columns:
        print("ERROR: Missing wrist_y")
        return None

    y = pose_df["wrist_y_smooth"].values.astype(float)
    n = len(y)

    # -----------------------------
    # Address + backswing start (your logic)
    # -----------------------------
    MIN_TOTAL_DROP = 0.2
    total_range = np.nanpercentile(y, 95) - np.nanpercentile(y, 5)
    total_range = max(float(total_range), 1e-6)
    significant_drop = max(0.30 * total_range, MIN_TOTAL_DROP)

    rs = rolling_std(y, win=20)
    max_search = min(n // 2, 250)
    stability_threshold = 0.06 * total_range
    min_len = 12

    stable_segments = []
    start = None
    length = 0
    for i in range(20, max_search):
        if rs[i] < stability_threshold:
            if start is None:
                start = i
            length += 1
        else:
            if start is not None and length >= min_len:
                stable_segments.append((start, i-1, length))
            start = None
            length = 0
    if start is not None and length >= min_len:
        stable_segments.append((start, max_search-1, length))

    if stable_segments:
        stable_segments.sort(key=lambda t: t[2], reverse=True)
        s0, s1, _ = stable_segments[0]
    else:
        s0, s1 = 0, min(20, max_search-1)

    address_idx = int(s0)
    address_y = float(np.nanmean(y[s0:s1+1]))

    local_drop = 0.02 * total_range
    confirm_window = 40
    backswing_start = int(s1)

    for i in range(s1, min(s1 + 80, n - confirm_window - 1)):
        if y[i] < address_y - local_drop:
            future_min = float(np.nanmin(y[i:i + confirm_window]))
            if (address_y - future_min) >= significant_drop:
                backswing_start = int(max(i - 5, s1))
                break

    if backswing_start == s1:
        dy0 = np.diff(y)
        for i in range(20, min(250, len(dy0) - 6)):
            if np.all(dy0[i:i+6] < -0.01 * total_range):
                backswing_start = int(i)
                break

    if debug:
        print(f"Address: frame {address_idx}, address_y: {address_y:.4f}")
        print(f"Backswing start: frame {backswing_start}")

    # -----------------------------
    # Backswing top (your peak/trough method from VSCode style)
    # -----------------------------
    search_window = min(140, n - backswing_start - 1)
    segment = y[backswing_start: backswing_start + search_window]

    seg_range = float(np.nanmax(segment) - np.nanmin(segment))
    prom = 0.15 * max(seg_range, 1e-6)
    troughs, _ = find_peaks(-segment, prominence=prom, distance=8)

    MIN_DROP_FROM_START = 0.2
    LOOKAHEAD = 25

    y_at_start = float(y[backswing_start])
    backswing_top = None

    if len(troughs) > 0:
        for t in troughs:
            trough_idx = int(backswing_start + t)
            y_at_trough = float(y[trough_idx])
            drop_from_start = y_at_start - y_at_trough
            if drop_from_start < MIN_DROP_FROM_START:
                continue

            j = min(t + LOOKAHEAD, len(segment) - 1)
            move_after = float(np.nanmax(segment[t:j+1]) - np.nanmin(segment[t:j+1]))
            if move_after >= 0.1:
                backswing_top = trough_idx
                break

    if backswing_top is None and len(troughs) > 0:
        valid = []
        for t in troughs:
            trough_idx = int(backswing_start + t)
            drop_from_start = y_at_start - float(y[trough_idx])
            if drop_from_start >= MIN_DROP_FROM_START:
                valid.append(t)
        if valid:
            best = int(valid[np.argmin([segment[t] for t in valid])])
            backswing_top = int(backswing_start + best)

    if backswing_top is None:
        min_idx = int(np.nanargmin(segment))
        y_at_min = float(segment[min_idx])
        drop_from_start = y_at_start - y_at_min
        if drop_from_start >= MIN_DROP_FROM_START:
            backswing_top = int(backswing_start + min_idx)
        else:
            backswing_top = int(backswing_start)

    if debug:
        print(f"Backswing top: frame {backswing_top}, y: {y[backswing_top]:.4f}")

    # -----------------------------
    # Impact wrist (your method)
    # -----------------------------
    search_start = int(backswing_top)
    search_end = int(min(backswing_top + 120, n))
    post = y[search_start:search_end]

    seg_range = float(np.nanmax(post) - np.nanmin(post))
    prom = 0.12 * max(seg_range, 1e-6)
    peaks, _ = find_peaks(post, prominence=prom, distance=8)

    if len(peaks) > 0:
        impact_wrist = int(search_start + peaks[0])
    else:
        impact_wrist = int(search_start + int(np.nanargmax(post)))

    impact = impact_wrist
    impact_method = "mediapipe_wrist"

    # -----------------------------
    # Impact refined with YOLO in +/- 5 frames
    # -----------------------------
    if club_data is not None and club_data.get("detection_rate", 0) > 10:
        impact_yolo, how = refine_impact_with_yolo_window(
            club_data, impact_wrist, window=IMPACT_WINDOW_FRAMES, debug=debug
        )
        if impact_yolo is not None:
            impact = int(np.clip(impact_yolo, 0, n - 1))
            impact_method = how

    if debug:
        print(f"Impact wrist={impact_wrist}, final={impact} [{impact_method}]")

    # -----------------------------
    # Finish (wrist speed stabilisation)
    # -----------------------------
    finish = n - 1
    if "wrist_x_smooth" in pose_df.columns:
        wrist_x = pose_df["wrist_x_smooth"].values.astype(float)
        wrist_y = pose_df["wrist_y_smooth"].values.astype(float)
        dt = 1.0 / fps
        vx = np.diff(wrist_x) / dt
        vy = np.diff(wrist_y) / dt
        speed = np.sqrt(vx*vx + vy*vy)

        post_speed = speed[min(impact, len(speed)-1):]
        if len(post_speed) >= 20:
            peak = float(np.nanmax(speed)) if len(speed) else 1.0
            thresh = 0.12 * max(peak, 1e-6)
            stable_len = 12
            count = 0
            for i in range(len(post_speed)):
                if post_speed[i] < thresh:
                    count += 1
                    if count >= stable_len:
                        finish = min(impact + i, n - 1)
                        break
                else:
                    count = 0

    print(f"\n{'='*50}")
    print("PHASE DETECTION RESULTS:")
    print(f"{'='*50}")
    print(f"  Address:         Frame {address_idx} ({address_idx/fps:.2f}s)")
    print(f"  Backswing Start: Frame {backswing_start} ({backswing_start/fps:.2f}s)")
    print(f"  Backswing Top:   Frame {backswing_top} ({backswing_top/fps:.2f}s)")
    print(f"  Impact (wrist):  Frame {impact_wrist} ({impact_wrist/fps:.2f}s)")
    print(f"  Impact (final):  Frame {impact} ({impact/fps:.2f}s) [{impact_method}]")
    print(f"  Finish:          Frame {finish} ({finish/fps:.2f}s)")
    print(f"{'='*50}")

    return {
        "address_idx": int(address_idx),
        "backswing_start_idx": int(backswing_start),
        "backswing_top_idx": int(backswing_top),
        "impact_wrist_idx": int(impact_wrist),
        "impact_idx": int(impact),
        "impact_method": str(impact_method),
        "finish_idx": int(finish),
        "fps": float(fps),
    }

# =============================================================================
# Visualization
# =============================================================================
def visualize_results(video_path, pose_df, club_data, phases, fps):
    print("\n" + "=" * 60)
    print("STEP 4: VISUALIZATION")
    print("=" * 60)

    phase_frames = {
        "Address": phases["address_idx"],
        "Backswing Top": phases["backswing_top_idx"],
        "Impact": phases["impact_idx"],
        "Finish": phases["finish_idx"],
    }

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for i, (name, idx) in enumerate(phase_frames.items()):
        img = extract_frame(video_path, idx)
        axes[i].imshow(img if img is not None else np.zeros((100,100,3), dtype=np.uint8))
        axes[i].set_title(f"{name}\nFrame {idx} ({idx/fps:.2f}s)", fontsize=12)
        axes[i].axis("off")
    plt.suptitle(f"Swing Phases (Impact: {phases['impact_method']})", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Wrist trajectory
    wrist_y = pose_df["wrist_y_smooth"].values.astype(float)
    frames = np.arange(len(wrist_y))

    plt.figure(figsize=(14,4))
    plt.plot(frames, wrist_y, linewidth=2, label="Wrist Y (MediaPipe)")
    plt.axvline(phases["address_idx"], linestyle="--", label="Address")
    plt.axvline(phases["backswing_top_idx"], linestyle="--", label="Backswing Top")
    plt.axvline(phases["impact_wrist_idx"], linestyle="--", label="Impact (wrist)")
    plt.axvline(phases["impact_idx"], linestyle="--", label="Impact (final)")
    plt.axvline(phases["finish_idx"], linestyle="--", label="Finish")
    plt.title("Wrist Trajectory (MediaPipe)")
    plt.xlabel("Frame")
    plt.ylabel("Wrist Y")
    plt.gca().invert_yaxis()
    plt.legend(loc="upper right")
    plt.show()

    # Club trajectory
    if club_data is not None:
        club_y = club_data["y_smooth"]
        plt.figure(figsize=(14,4))
        plt.plot(np.arange(len(club_y)), club_y, linewidth=2, label="Club Y (YOLO)")
        plt.axvline(phases["backswing_top_idx"], linestyle="--", label="Backswing Top")
        plt.axvline(phases["impact_wrist_idx"], linestyle="--", label="Impact (wrist)")
        plt.axvline(phases["impact_idx"], linestyle="--", label="Impact (final)")
        plt.title(f"Club Trajectory (YOLO) - Detection rate: {club_data['detection_rate']:.1f}%")
        plt.xlabel("Frame")
        plt.ylabel("Club Y (pixels)")
        plt.legend(loc="upper right")
        plt.show()

# =============================================================================
# MAIN
# =============================================================================
print("=" * 60)
print("GOLF SWING ANALYSIS (MediaPipe + YOLO impact refine ±5 frames)")
print("=" * 60)
print("Video:", VIDEO_PATH)
print("YOLO model:", CLUB_MODEL_PATH)
print("=" * 60)

pose_df, fps = extract_pose_mediapipe(VIDEO_PATH)
club_data = extract_club_trajectory(VIDEO_PATH, CLUB_MODEL_PATH, conf=CONF, keypoint_idx=CLUB_KEYPOINT_IDX)

phases = detect_phases(pose_df, club_data, fps, debug=True)
visualize_results(VIDEO_PATH, pose_df, club_data, phases, fps)